## RAG pipeline - Data ingestion to vector DB pipeline

In [6]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [7]:
## read all the PDFs inside the directory

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: python-cheatsheet.pdf


Ignoring wrong pointing object 8 0 (offset 0)


  ✓ Loaded 3 pages

Processing: sample.pdf
  ✓ Loaded 1 pages

Processing: sample-local-pdf.pdf
  ✓ Loaded 3 pages

Total documents loaded: 7


In [8]:
all_pdf_documents

[Document(metadata={'producer': 'Created by Marp', 'creator': 'Created by Marp', 'creationdate': '2025-09-17T09:44:27+00:00', 'title': 'Python Cheat Sheet', 'moddate': '2025-09-17T09:44:27+00:00', 'subject': 'Continue your learning journey and become a Python expert at realpython.com/start-here', 'source': '../data/pdf_files/python-cheatsheet.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'python-cheatsheet.pdf', 'file_type': 'pdf'}, page_content='Real PythonPocket Reference\nVisit realpython.com to\nturbocharge your\nPython learning with\nin-depth tutorials,\nreal-world examples,\nand expert guidance.\nGetting Started\nFollow these guides to kickstart your Python journey:\nrealpython.com/what-can-i-do-with-python\nrealpython.com/installing-python\nrealpython.com/python-first-steps\nStart the Interactive Shell\n$ python\nQuit the Interactive Shell\n>>> exit()\nRun a Script\n$ python my_script.py\nRun a Script in Interactive Mode\n$ python -i my_script.py\nLearn Mo

In [9]:
## text splitting and get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [10]:
chunks=split_documents(all_pdf_documents)
chunks

Split 7 documents into 39 chunks

Example chunk:
Content: Real PythonPocket Reference
Visit realpython.com to
turbocharge your
Python learning with
in-depth tutorials,
real-world examples,
and expert guidance.
Getting Started
Follow these guides to kickstart...
Metadata: {'producer': 'Created by Marp', 'creator': 'Created by Marp', 'creationdate': '2025-09-17T09:44:27+00:00', 'title': 'Python Cheat Sheet', 'moddate': '2025-09-17T09:44:27+00:00', 'subject': 'Continue your learning journey and become a Python expert at realpython.com/start-here', 'source': '../data/pdf_files/python-cheatsheet.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'python-cheatsheet.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Created by Marp', 'creator': 'Created by Marp', 'creationdate': '2025-09-17T09:44:27+00:00', 'title': 'Python Cheat Sheet', 'moddate': '2025-09-17T09:44:27+00:00', 'subject': 'Continue your learning journey and become a Python expert at realpython.com/start-here', 'source': '../data/pdf_files/python-cheatsheet.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'python-cheatsheet.pdf', 'file_type': 'pdf'}, page_content='Real PythonPocket Reference\nVisit realpython.com to\nturbocharge your\nPython learning with\nin-depth tutorials,\nreal-world examples,\nand expert guidance.\nGetting Started\nFollow these guides to kickstart your Python journey:\nrealpython.com/what-can-i-do-with-python\nrealpython.com/installing-python\nrealpython.com/python-first-steps\nStart the Interactive Shell\n$ python\nQuit the Interactive Shell\n>>> exit()\nRun a Script\n$ python my_script.py\nRun a Script in Interactive Mode\n$ python -i my_script.py\nLearn Mo

### embedding & vector store DB

In [11]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

/Users/lucas3.0/Desktop/AI Agents/03_RAG/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 23140.68it/s]


Model loaded successfully. Embedding dimension: 384


/var/folders/mk/2xghfztx4sn2cwryxkm98b900000gn/T/ipykernel_82488/221633724.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### VectorStore

In [13]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [14]:
chunks

[Document(metadata={'producer': 'Created by Marp', 'creator': 'Created by Marp', 'creationdate': '2025-09-17T09:44:27+00:00', 'title': 'Python Cheat Sheet', 'moddate': '2025-09-17T09:44:27+00:00', 'subject': 'Continue your learning journey and become a Python expert at realpython.com/start-here', 'source': '../data/pdf_files/python-cheatsheet.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'python-cheatsheet.pdf', 'file_type': 'pdf'}, page_content='Real PythonPocket Reference\nVisit realpython.com to\nturbocharge your\nPython learning with\nin-depth tutorials,\nreal-world examples,\nand expert guidance.\nGetting Started\nFollow these guides to kickstart your Python journey:\nrealpython.com/what-can-i-do-with-python\nrealpython.com/installing-python\nrealpython.com/python-first-steps\nStart the Interactive Shell\n$ python\nQuit the Interactive Shell\n>>> exit()\nRun a Script\n$ python my_script.py\nRun a Script in Interactive Mode\n$ python -i my_script.py\nLearn Mo

In [15]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 39 texts...


Batches: 100%|██████████| 2/2 [00:02<00:00,  1.12s/it]

Generated embeddings with shape: (39, 384)
Adding 39 documents to vector store...
Successfully added 39 documents to vector store
Total documents in collection: 39


## Retriever Pipeline From VectorStore

In [18]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

rag_retriever

In [20]:
rag_retriever.retrieve("tell about Imports & Modules in python cheatsheet")

Retrieving documents for query: 'tell about Imports & Modules in python cheatsheet'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.19it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


[{'id': 'doc_7bb829dc_16',
  'content': '# Import all (not recommended)\nfrom math import *\nPackage Imports\n# Import from package\nimport package.module\nfrom package import module\nfrom package.subpackage import module\n# Import specific items\nfrom package.module import function, Class\nfrom package.module import name as alias\nLearn More on realpython.com/search:\nimport ∙ modules ∙ packages\nVirtual Environments\nVirtual Environments are often called “venv”\nUse venvs to isolate project packages from the system-wide Python packages\nCreate Virtual Environment\n$ python -m venv .venv\nActivate Virtual Environment (Windows)\nPS> .venv\\Scripts\\activate\nActivate Virtual Environment (Linux & macOS)\n$ source .venv/bin/activate\nDeactivate Virtual Environment\n(.venv) $ deactivate\nLearn More on realpython.com/search:\nvirtual environment ∙ venv\nPackages\nThe oﬀicial third-party package repository is the Python Package Index (PyPI)\nInstall Packages\n$ python -m pip install request

In [21]:
rag_retriever.retrieve("tell everything about python")

Retrieving documents for query: 'tell everything about python'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.00it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


[{'id': 'doc_6b7611aa_0',
  'content': 'Real PythonPocket Reference\nVisit realpython.com to\nturbocharge your\nPython learning with\nin-depth tutorials,\nreal-world examples,\nand expert guidance.\nGetting Started\nFollow these guides to kickstart your Python journey:\nrealpython.com/what-can-i-do-with-python\nrealpython.com/installing-python\nrealpython.com/python-first-steps\nStart the Interactive Shell\n$ python\nQuit the Interactive Shell\n>>> exit()\nRun a Script\n$ python my_script.py\nRun a Script in Interactive Mode\n$ python -i my_script.py\nLearn More on realpython.com/search:\ninterpreter ∙ run a script ∙ command line\nComments\nAlways add a space after the #\nUse comments to explain “why” of your code\nWrite Comments\n# This is a comment\n# print("This code will not run.")\nprint("This will run.")  # Comments are ignored by Python\nLearn More on realpython.com/search:\ncomment ∙ documentation\nData Types\nPython is dynamically typed\nUse None to represent missing or option